In [3]:
import numpy as np
import pandas as pd

In [6]:
df_sales = pd.read_csv('advanced_pandas_sales_dataset (1).csv')
df_sales.head()

,OrderID,OrderDate,Region,Category,Product,Quantity,UnitPrice,Revenue
0,1001,2025-01-05,East,Electronics,Laptop,6,700,4200
1,1002,2025-01-05,West,Office Supplies,Pen,5,910,4550
2,1003,2025-01-05,North,Office Supplies,Printer Paper,1,140,140
3,1004,2025-01-05,South,Furniture,Cabinet,4,720,2880
4,1005,2025-01-10,East,Furniture,Desk,5,160,800


In [8]:
category_revenue = df_sales.groupby('Category')['Revenue'].sum().reset_index()
category_revenue

,Category,Revenue
0,Electronics,130950
1,Furniture,134590
2,Office Supplies,103700


In [12]:
# total items sold by region
region_metrics = df_sales.groupby('Region').agg(
    Total_Units_Sold = ('Quantity', 'sum'),
    Average_Item_Price = ('UnitPrice', 'mean')
).reset_index()
region_metrics


,Region,Total_Units_Sold,Average_Item_Price
0,East,193,462.333333
1,North,161,552.000000
2,South,141,621.333333
3,West,200,523.666667


In [13]:
managers_data = {
    'Region': ['North', 'South', 'East', 'West'],
    'Regional_Director': ['Alice Cooper', 'Bob Marley', 'Charlie Chaplin', 'David Beckham']
}
df_managers = pd.DataFrame(managers_data)

In [16]:
# Merge the main sales log with the managers database using the 'Region' column as the key
df_enriched = pd.merge(df_sales,df_managers, on='Region', how='left')

print(df_enriched[['OrderID', 'Region', 'Category', 'Revenue', 'Regional_Director']].head())

   OrderID Region         Category  Revenue Regional_Director
0     1001   East      Electronics     4200   Charlie Chaplin
1     1002   West  Office Supplies     4550     David Beckham
2     1003  North  Office Supplies      140      Alice Cooper
3     1004  South        Furniture     2880        Bob Marley
4     1005   East        Furniture      800   Charlie Chaplin


In [20]:
revenue_pivot = pd.pivot_table(
    df_sales,
    values='Revenue',
    index='Region',
    columns='Category',
    aggfunc='sum',
)

print('--- Revenue Pivot Table (Revenue vs Category) ---')
print(revenue_pivot)

--- Revenue Pivot Table (Revenue vs Category) ---
Category  Electronics  Furniture  Office Supplies
Region                                           
East            34190      37820            16300
North           14260      27430            42080
South           40900      29680            23530
West            41600      39660            21790


In [21]:
order_count_crosstab = pd.crosstab(
    index = df_sales['Region'],
    columns = df_sales['Category'],
)

print('--- Order Frequency Crosstab (Transaction Count) ---')
print(order_count_crosstab)

--- Order Frequency Crosstab (Transaction Count) ---
Category  Electronics  Furniture  Office Supplies
Region                                           
East               10         14                6
North               8          7               15
South              11         11                8
West               10         15                5


**Time Series**

In [24]:
print('Original OrderDate column data type:', df_sales['OrderDate'])

Original OrderDate column data type: 0      2025-01-05
1      2025-01-05
2      2025-01-05
3      2025-01-05
4      2025-01-10
          ...    
115    2025-06-20
116    2025-06-25
117    2025-06-25
118    2025-06-25
119    2025-06-25
Name: OrderDate, Length: 120, dtype: str


In [26]:
df_sales['OrderDate'] = pd.to_datetime(df_sales['OrderDate'])
print('Converted OrderDate column data type:', df_sales['OrderDate'])

Converted OrderDate column data type: 0     2025-01-05
1     2025-01-05
2     2025-01-05
3     2025-01-05
4     2025-01-10
         ...    
115   2025-06-20
116   2025-06-25
117   2025-06-25
118   2025-06-25
119   2025-06-25
Name: OrderDate, Length: 120, dtype: datetime64[us]


In [29]:
df_sales['Month_Name'] = df_sales['OrderDate'].dt.strftime("%B") # For full month name
df_sales['Day_of_Week'] = df_sales['OrderDate'].dt.day_name() # For day of the week name


In [31]:
monthly_trends = df_sales.groupby(df_sales['OrderDate'].dt.to_period('M'))['Revenue'].sum().reset_index()
print(monthly_trends)

  OrderDate  Revenue
0   2025-01    52320
1   2025-02    67440
2   2025-03    65680
3   2025-04    68950
4   2025-05    62160
5   2025-06    52690


In [32]:
df_election = pd.read_csv('election_data.csv')
df_district = pd.read_csv('district_data (1).csv')
df_province = pd.read_csv('province_data.csv')

In [34]:
print(df_election.shape)
print(df_district.shape)
print(df_province.shape)

(3406, 8)
(77, 3)
(7, 2)


In [41]:
print("Election Sample Key\n", df_election[['candidate_name','district_id']].head(2))
print("District Sample Key\n", df_district[['district_name','district_id','province_id']].head(2))
print("Province Sample Key\n", df_province.head(2))

Election Sample Key
       candidate_name  district_id
0  Dharmraj Guragain            4
1     Pyare Lal Rana           71
District Sample Key
   district_name  district_id  province_id
0     Taplejung            1            1
1     Panchthar            2            1
Province Sample Key
    province_id province_name
0            1         Koshi
1            2       Madhesh


In [45]:
df_candidate = pd.merge(
    left = df_election,
    right = df_district,
    on = 'district_id',
    how = 'inner'
)
print(df_candidate.shape)
print(df_candidate[['candidate_name','party','district_name','votes']].head(3))

(3406, 10)
            candidate_name                  party district_name  votes
0        Dharmraj Guragain     People First Party         Jhapa     34
1           Pyare Lal Rana            Independent       Kailali   1755
2  Bidya Shrestha Maharjan  Shram Sanskriti Party     Makwanpur   1024


In [48]:
df_district_left = pd.merge(
    left = df_candidate,
    right = df_province,
    on = 'province_id',
    how = 'left'
)
print(df_district_left.shape)
print(df_district_left[['district_name','province_name','province_id']].head(3))

(3406, 11)
  district_name  province_name  province_id
0         Jhapa          Koshi            1
1       Kailali  Sudurpashchim            7
2     Makwanpur        Bagmati            3
